# Entity Action V2 - Legal-Action Candidate Training (Colab)

This notebook adapts the v1 entity training flow to the new candidate-conditioned `entity_action_v2` model.

What is different from v1:
- the policy target is the chosen legal candidate index, not a global action-vocabulary class
- the model scores move and switch candidates directly
- offline replay training currently falls back to observed moves plus switch slots when full simulator request legality is unavailable

Current scope:
- policy head: enabled
- value head: optional and enabled below by default
- transition and sequence auxiliaries: not wired into v2 yet


## Setup: GPU, Repo, Dependencies, Data


In [ ]:
# 1. Verify GPU
import subprocess

gpu_available = False
try:
    r = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
        capture_output=True, text=True, timeout=5
    )
    if r.returncode == 0:
        print('GPU:', r.stdout.strip())
        gpu_available = True
except (FileNotFoundError, subprocess.TimeoutExpired):
    pass

import tensorflow as tf
print('TensorFlow:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print('TF GPUs:', [g.name for g in gpus])
    gpu_available = True
else:
    print('TF: CPU only')

if not gpu_available:
    print()
    print('WARNING: No GPU detected.')
    print('Colab: Runtime -> Change runtime type -> T4 GPU, then restart.')


In [ ]:
# 2. Clone repo
import os

REPO_URL = 'https://github.com/AlterProgramming/Pokemon-Showdown-Sim.git'
# Set this to the branch that contains EntityModelV2 / EntityTensorizationV2.
BRANCH = 'codex/claude-entity-sequence-plan'
REPO_DIR = '/content/Pokemon-Showdown-Sim'

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
!git log --oneline -3
print()
print('If this branch does not contain EntityModelV2.py and EntityTensorizationV2.py, update BRANCH first.')


In [ ]:
# 3. Install dependencies
import importlib
import tensorflow as tf

print('TF:', tf.__version__)
!pip install -q kagglehub

try:
    import keras
    if int(keras.__version__.split('.')[0]) < 3:
        print('Upgrading keras to 3.x...')
        !pip install -q 'keras>=3.0'
        importlib.invalidate_caches()
    else:
        print('keras', keras.__version__, 'OK')
except ImportError:
    !pip install -q 'keras>=3.0'

print('Done. If first run: Runtime -> Restart session, then re-run from cell 1.')


In [ ]:
# 4. Download battle data
import glob
import kagglehub
import os

DATASET = 'thephilliplin/pokemon-showdown-battles-gen9-randbats'
print('Downloading', DATASET, '...')
data_path = kagglehub.dataset_download(DATASET)

json_files = glob.glob(os.path.join(data_path, '*.json'))
print(f'Dataset path : {data_path}')
print(f'JSON files   : {len(json_files):,}')


## Training Configuration


In [ ]:
# 5. Training configuration
import os
from datetime import datetime

_ts = datetime.now().strftime('%Y%m%d_%H%M')
MODEL_NAME = f'entity_action_v2_{_ts}'
OUTPUT_DIR = '/content/training_artifacts_v2'

MAX_BATTLES   = 5000
VAL_RATIO     = 0.2
EPOCHS        = 20
BATCH_SIZE    = 256
LEARNING_RATE = 0.001
PATIENCE      = 3
HIDDEN_DIM    = 256
DEPTH         = 3
DROPOUT       = 0.1
TOKEN_EMBED_DIM = 24
MAX_CANDIDATES  = 10
PREDICT_VALUE   = True
VALUE_WEIGHT    = 0.25
SEED            = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)

print('=' * 70)
print('ENTITY ACTION V2 CONFIGURATION')
print('=' * 70)
print(f'  Model name     : {MODEL_NAME}')
print(f'  Output dir     : {OUTPUT_DIR}')
print(f'  Max battles    : {MAX_BATTLES}')
print(f'  Epochs         : {EPOCHS}  (early stop patience={PATIENCE})')
print(f'  Batch size     : {BATCH_SIZE}   LR={LEARNING_RATE}')
print(f'  Architecture   : hidden={HIDDEN_DIM}  depth={DEPTH}  dropout={DROPOUT}  embed={TOKEN_EMBED_DIM}')
print(f'  Max candidates : {MAX_CANDIDATES}')
print(f'  Value head     : {PREDICT_VALUE}  (w={VALUE_WEIGHT})')
print('=' * 70)


## Train V2


In [ ]:
# 6. Train via the dedicated CLI trainer
import os
import re
import subprocess
import sys

env = os.environ.copy()
env['PYTHONPATH'] = REPO_DIR + os.pathsep + env.get('PYTHONPATH', '')

cmd = [
    sys.executable, '-u', 'train_entity_action_v2.py', data_path,
    '--model-name', MODEL_NAME,
    '--output-dir', OUTPUT_DIR,
    '--max-battles', str(MAX_BATTLES),
    '--val-ratio', str(VAL_RATIO),
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--learning-rate', str(LEARNING_RATE),
    '--hidden-dim', str(HIDDEN_DIM),
    '--depth', str(DEPTH),
    '--dropout', str(DROPOUT),
    '--token-embed-dim', str(TOKEN_EMBED_DIM),
    '--max-candidates', str(MAX_CANDIDATES),
    '--patience', str(PATIENCE),
    '--seed', str(SEED),
]
if PREDICT_VALUE:
    cmd += ['--predict-value', '--value-weight', str(VALUE_WEIGHT)]

print('Running:')
print(' '.join(cmd))
print()

proc = subprocess.Popen(
    cmd,
    cwd=REPO_DIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

epoch_re = re.compile(r'^Epoch\s+\d+/\d+')
signal_kws = ('entity_v2_training_start', 'final_ingest:', 'entity_v2_training_complete:')
captured = []
for line in proc.stdout:
    captured.append(line)
    line = line.rstrip('\n')
    if epoch_re.search(line) or any(kw in line for kw in signal_kws) or 'val_' in line or 'loss:' in line:
        print(line)

return_code = proc.wait()
if return_code != 0:
    print(''.join(captured[-200:]))
    raise RuntimeError(f'train_entity_action_v2.py exited with status {return_code}')

print()
print('Training finished successfully.')


In [ ]:
# 7. Inspect saved artifacts
import json
import os

artifacts = sorted(os.listdir(OUTPUT_DIR))
print('=' * 70)
print('ARTIFACTS')
print('=' * 70)
for f in artifacts:
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    size_str = f'{size/(1024*1024):.2f} MB' if size > 1024*1024 else f'{size/1024:.1f} KB'
    print(f'  {f:60s}  {size_str:>10}')

meta_path = os.path.join(OUTPUT_DIR, f'training_metadata_{MODEL_NAME}.json')
meta = json.load(open(meta_path))
print()
print('KEY METADATA')
print('-' * 70)
for key in [
    'model_name', 'family_id', 'training_regime', 'information_policy',
    'train_examples', 'val_examples', 'epochs_completed',
    'max_candidates', 'predict_value'
]:
    print(f'  {key:24s}: {meta.get(key)}')


In [ ]:
# ── 13. Push artifacts to GCS ─────────────────────────────────────────────
# Uploads trained artifacts to gs://artifacts-model-serving/artifacts/{family}/{run_id}/
# Uses your logged-in Colab Google account — no secrets required.
import os, glob

BUCKET     = 'artifacts-model-serving'
GCS_PREFIX = f'artifacts/entity_action_v2/{RUN_ID}'

try:
    from google.colab import auth
    from google.cloud import storage

    auth.authenticate_user()  # one-time browser prompt per Colab session
    client = storage.Client(project='pokemon-battle-agent-494015')
    bucket = client.bucket(BUCKET)

    pushed = []
    for local_path in glob.glob(os.path.join(OUTPUT_DIR, '*')):
        if not os.path.isfile(local_path):
            continue
        fname     = os.path.basename(local_path)
        blob_name = f'{GCS_PREFIX}/{fname}'
        bucket.blob(blob_name).upload_from_filename(local_path)
        pushed.append(blob_name)
        print(f'  ✓ {blob_name}')

    print(f'\nPushed {len(pushed)} files → gs://{BUCKET}/{GCS_PREFIX}/')
    print('Run /gcs-push locally to sync model_registry.json.')

except Exception as e:
    print(f'GCS push failed: {e}')
